# Dinamic System Modelling: Pemodelan dan Simulasi Suhu Ruangan dengan AC
**Nama:** Soni Moch Leviansyah  
**NIM:** 301240030  
**Mata kuliah:** Pemodelan Sistem Dinamik

Model utama: $\frac{dT}{dt}=\frac{T_{out}-T}{\tau}-k\,u(t)$

## 0. Instalasi library

In [ ]:
!pip install gekko -q

## Program 1 - Suhu ruangan dengan AC dinyalakan jam 2-7 (ODEINT)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

# model suhu ruangan dengan AC
def ruangan(T, t, k, u):
    T_out = 32.0     # suhu udara luar (C)
    tau = 2.0        # konstanta waktu ruangan (jam)
    # hitung turunan suhu ruangan
    dTdt = (T_out - T) / tau - k * u
    return dTdt

# rentang waktu simulasi 10 jam, setiap 0.1 jam
ts = np.linspace(0, 10, 101)

# operasi AC
k = 0.07                # koefisien AC (C/jam per % daya)
u = np.zeros(101)       # u = daya AC (%)
u[20:70] = 100.0        # AC menyala antara jam ke-2 dan ke-7

# kondisi awal suhu ruangan
T0 = 30.0

# penyimpan hasil
z = np.zeros(101)
z[0] = T0

# simulasi dengan ODEINT
for i in range(100):
    daya = u[i]
    y = odeint(ruangan, T0, [0, 0.1], args=(k, daya))
    T0 = y[-1][0]          # ambil titik terakhir
    z[i + 1] = T0          # simpan suhu untuk plot

# plot hasil
plt.figure()
plt.subplot(2, 1, 1)
plt.plot(ts, z, 'b-')
plt.ylabel('Suhu Ruangan $T(t)$, (C)')
plt.subplot(2, 1, 2)
plt.plot(ts, u, 'r--')
plt.ylabel('Daya AC $u(t)$, (%)')
plt.xlabel('$t$, (jam)')
plt.show()


## Program 2 - AC dikendalikan termostat 24-26 C (ODEINT)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

# model suhu ruangan dengan AC
def ruangan(T, t, u):
    T_out = 32.0
    tau = 2.0
    k = 0.07
    return (T_out - T) / tau - k * u

# batas termostat
T_on = 26.0       # AC menyala jika T > 26 C
T_off = 24.0      # AC mati jika T < 24 C

# waktu simulasi 10 jam, setiap 0.01 jam
dt = 0.01
ts = np.arange(0, 10 + dt, dt)

# penyimpan hasil
T = np.zeros(len(ts))
u = np.zeros(len(ts))
T[0] = 30.0                 # suhu awal
status = 0.0                # 0 = AC mati, 100 = AC menyala

# simulasi dengan ODEINT
for i in range(1, len(ts)):
    if T[i - 1] > T_on:
        status = 100.0
    elif T[i - 1] < T_off:
        status = 0.0
    u[i] = status
    y = odeint(ruangan, T[i - 1], [0, dt], args=(status,))
    T[i] = y[-1][0]

# plot hasil
plt.figure()
plt.subplot(2, 1, 1)
plt.plot(ts, T, 'b-')
plt.axhline(T_on, color='r', linestyle='--')
plt.axhline(T_off, color='g', linestyle='--')
plt.ylabel('Suhu Ruangan $T(t)$, (C)')
plt.subplot(2, 1, 2)
plt.plot(ts, u, 'r-')
plt.ylabel('Daya AC $u(t)$, (%)')
plt.xlabel('$t$, (jam)')
plt.show()

print('AC menyala total = %.2f jam' % (u.sum() * dt / 100))


## Program 3 - Contoh 1: peluruhan selisih suhu (ODEINT)

In [ ]:
from scipy.integrate import odeint
import numpy as np
import matplotlib.pyplot as plt

# fungsi yang mengembalikan dy/dt
def model(y, t):
    k = 0.5
    dydt = -k * y
    return dydt

# kondisi awal
y0 = 10

# titik waktu
t = np.linspace(0, 10)

# selesaikan ODE
y = odeint(model, y0, t)

# plot hasil
plt.plot(t, y)
plt.xlabel('$t$, (jam)')
plt.ylabel('$y(t)$, (C)')
plt.show()


## Program 4 - Contoh 1 (GEKKO)

In [ ]:
import numpy as np
from gekko import GEKKO
import matplotlib.pyplot as plt

m = GEKKO(remote=False)           # buat model GEKKO
k = 0.5                           # konstanta
y = m.Var(10.0)                   # buat variabel GEKKO
m.Equation(y.dt() == -k * y)      # buat persamaan GEKKO
m.time = np.linspace(0, 10)       # titik waktu

# selesaikan ODE
m.options.IMODE = 4
m.solve(disp=False)

# plot hasil
plt.plot(m.time, y)
plt.xlabel('$t$, (jam)')
plt.ylabel('$y(t)$, (C)')
plt.show()


## Program 5 - Contoh 2: variasi k dengan args (ODEINT)

In [ ]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

# fungsi yang mengembalikan dy/dt
def model(y, t, k):
    dydt = -k * y
    return dydt

# kondisi awal
y0 = 10

# titik waktu
t = np.linspace(0, 10)

# selesaikan ODE
k = 0.25
y1 = odeint(model, y0, t, args=(k,))
k = 0.5
y2 = odeint(model, y0, t, args=(k,))
k = 1.0
y3 = odeint(model, y0, t, args=(k,))

# plot hasil
plt.plot(t, y1, 'r-', label=r'$k=0.25$')
plt.plot(t, y2, 'b--', label=r'$k=0.5$')
plt.plot(t, y3, 'g:', label=r'$k=1.0$')
plt.xlabel('$t$, (jam)')
plt.ylabel('$y(t)$, (C)')
plt.legend()
plt.show()


## Program 6 - Contoh 2 (GEKKO)

In [ ]:
import numpy as np
from gekko import GEKKO
import matplotlib.pyplot as plt

m = GEKKO(remote=False)           # buat model GEKKO
k = m.Param()                     # konstanta
y = m.Var(10.0)                   # buat variabel GEKKO
m.Equation(y.dt() == -k * y)      # buat persamaan GEKKO
m.time = np.linspace(0, 10)       # titik waktu

# selesaikan ODE dan plot
m.options.IMODE = 4
m.options.TIME_SHIFT = 0

k.value = 0.25
m.solve(disp=False)
plt.plot(m.time, y.value, 'r-', linewidth=2, label='k=0.25')

k.value = 0.5
m.solve(disp=False)
plt.plot(m.time, y.value, 'b--', linewidth=2, label='k=0.5')

k.value = 1.0
m.solve(disp=False)
plt.plot(m.time, y.value, 'g:', linewidth=2, label='k=1.0')

plt.xlabel('$t$, (jam)')
plt.ylabel('$y(t)$, (C)')
plt.legend()
plt.show()


## Program 7 - Contoh 3: input step AC (ODEINT)

In [ ]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

# fungsi yang mengembalikan dT/dt
def model(T, t):
    # u melompat dari 0 ke 100 pada t = 2
    if t < 2.0:
        u = 0
    else:
        u = 100
    dTdt = (32.0 - T) / 2.0 - 0.07 * u
    return dTdt

# kondisi awal
T0 = 30

# titik waktu
t = np.linspace(0, 10, 1000)

# selesaikan ODE
T = odeint(model, T0, t, tcrit=[2.0])

# plot hasil
plt.subplot(2, 1, 1)
plt.plot(t, T, 'r-')
plt.ylabel('Suhu $T(t)$, (C)')
plt.subplot(2, 1, 2)
plt.plot([0, 2, 2, 10], [0, 0, 100, 100], 'b-')
plt.ylabel('Daya AC $u(t)$, (%)')
plt.xlabel('$t$, (jam)')
plt.show()


## Program 8 - Contoh 3 (GEKKO)

In [ ]:
import numpy as np
from gekko import GEKKO
import matplotlib.pyplot as plt

m = GEKKO(remote=False)                  # buat model GEKKO
m.time = np.linspace(0, 10, 1001)        # titik waktu

# buat parameter GEKKO (step 0 ke 100 pada t = 2)
ustep = np.zeros(1001)
ustep[201:] = 100.0
u = m.Param(value=ustep)

T = m.Var(30.0)                          # buat variabel GEKKO
m.Equation(2 * T.dt() == (32.0 - T) - 2 * 0.07 * u)   # buat persamaan GEKKO

# selesaikan ODE
m.options.IMODE = 4
m.solve(disp=False)

# plot hasil
plt.subplot(2, 1, 1)
plt.plot(m.time, T, 'r-')
plt.ylabel('Suhu $T(t)$, (C)')
plt.subplot(2, 1, 2)
plt.plot(m.time, ustep, 'b-')
plt.ylabel('Daya AC $u(t)$, (%)')
plt.xlabel('$t$, (jam)')
plt.show()


## Program 9 - Contoh 4: dua persamaan ekivalen (ODEINT)

In [ ]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

# fungsi yang mengembalikan dz/dt
def model(z, t):
    dxdt = -6.0 * np.exp(-0.5 * t)
    dydt = -0.5 * z[1] + 9.0
    dzdt = [dxdt, dydt]
    return dzdt

# kondisi awal
z0 = [30, 30]

# titik waktu
t = np.linspace(0, 10)

# selesaikan ODE
z = odeint(model, z0, t)

# plot hasil
plt.plot(t, z[:, 0], 'b-', label=r'$\frac{dx}{dt}=-6\,\exp(-0.5t)$')
plt.plot(t, z[:, 1], 'r--', label=r'$\frac{dy}{dt}=-0.5y+9$')
plt.ylabel('$x(t),\\; y(t)$, (C)')
plt.xlabel('$t$, (jam)')
plt.legend(loc='best')
plt.show()


## Program 10 - Contoh 4 (GEKKO)

In [ ]:
import numpy as np
from gekko import GEKKO
import matplotlib.pyplot as plt

m = GEKKO(remote=False)           # buat model GEKKO
m.time = np.linspace(0, 10)       # titik waktu

t = m.Var(0.0)                    # buat waktu sebagai variabel
x = m.Var(30.0)                   # buat variabel x
y = m.Var(30.0)                   # buat variabel y

# buat persamaan GEKKO
m.Equation(t.dt() == 1)
m.Equation(x.dt() == -6 * m.exp(-0.5 * t))
m.Equation(y.dt() == -0.5 * y + 9)

# selesaikan ODE
m.options.IMODE = 4
m.options.NODES = 3
m.solve(disp=False)

# plot hasil
plt.plot(t, x, 'b-', label=r'$dx/dt=-6e^{-0.5t}$')
plt.plot(t, y, 'r--', label=r'$dy/dt=-0.5y+9$')
plt.ylabel('respon (C)')
plt.xlabel('waktu (jam)')
plt.legend(loc='best')
plt.show()


## Program 11 - Contoh 5: model 2 state udara-dinding (ODEINT)

In [ ]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

# fungsi yang mengembalikan dz/dt
def model(z, t, u):
    T = z[0]                          # suhu udara ruangan
    W = z[1]                          # suhu dinding
    dTdt = ((W - T) - 0.5 * 0.12 * u) / 0.5
    dWdt = (32.0 - W) / 4.0
    dzdt = [dTdt, dWdt]
    return dzdt

# kondisi awal
z0 = [30, 31]

# jumlah titik waktu
n = 401

# titik waktu
t = np.linspace(0, 10, n)

# input step: AC menyala 100% mulai jam ke-2
u = np.zeros(n)
u[80:] = 100.0

# penyimpan hasil
T = np.empty_like(t)
W = np.empty_like(t)
T[0] = z0[0]
W[0] = z0[1]

# selesaikan ODE
for i in range(1, n):
    tspan = [t[i - 1], t[i]]
    z = odeint(model, z0, tspan, args=(u[i - 1],))
    T[i] = z[1][0]
    W[i] = z[1][1]
    z0 = z[1]

# plot hasil
plt.plot(t, u / 100 * 10 + 20, 'g:', label=r'$u(t)$ (skala)')
plt.plot(t, T, 'b-', label=r'$T(t)$ udara')
plt.plot(t, W, 'r--', label=r'$W(t)$ dinding')
plt.ylabel('Suhu (C)')
plt.xlabel('$t$, (jam)')
plt.legend(loc='best')
plt.show()


## Program 12 - Contoh 5 (GEKKO)

In [ ]:
import numpy as np
from gekko import GEKKO
import matplotlib.pyplot as plt

m = GEKKO(remote=False)                  # buat model GEKKO
m.time = np.linspace(0, 10, 401)         # titik waktu

# buat parameter GEKKO (step 0 ke 100 pada t = 2)
ustep = np.zeros(401)
ustep[81:] = 100.0
u = m.Param(value=ustep)

# buat variabel GEKKO
T = m.Var(30.0)
W = m.Var(31.0)

# buat persamaan GEKKO
m.Equation(0.5 * T.dt() == (W - T) - 0.5 * 0.12 * u)
m.Equation(4 * W.dt() == 32.0 - W)

# selesaikan ODE
m.options.IMODE = 4
m.solve(disp=False)

# plot hasil
plt.plot(m.time, ustep / 100 * 10 + 20, 'g:', label=r'$u(t)$ (skala)')
plt.plot(m.time, T, 'b-', label=r'$T(t)$ udara')
plt.plot(m.time, W, 'r--', label=r'$W(t)$ dinding')
plt.xlabel('$t$, (jam)')
plt.ylabel('Suhu (C)')
plt.legend(loc='best')
plt.show()


## Program 13 - Studi kasus: setpoint termostat dan energi AC (ODEINT)

In [ ]:
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

tau = 2.0          # konstanta waktu ruangan (jam)
k = 0.07           # koefisien pendinginan AC (C/jam per %)
P_ac = 0.9         # daya listrik AC saat menyala (kW)

# suhu udara luar berubah sepanjang hari (maksimum jam 15.00)
def T_luar(t):
    return 30.0 + 4.0 * np.sin(2 * np.pi * (t - 9.0) / 24.0)

# model suhu ruangan
def ruangan(T, t, u):
    return (T_luar(t) - T) / tau - k * u

# waktu simulasi 24 jam, setiap 0.01 jam
dt = 0.01
ts = np.arange(0, 24 + dt, dt)

# pilihan setpoint termostat: (batas OFF, batas ON)
pilihan = [(21, 23), (23, 25), (25, 27)]
warna = ['b', 'g', 'r']

plt.figure(figsize=(8, 6))
for (T_off, T_on), w in zip(pilihan, warna):
    T = np.zeros(len(ts))
    u = np.zeros(len(ts))
    T[0] = 30.0
    status = 0.0
    for i in range(1, len(ts)):
        if T[i - 1] > T_on:
            status = 100.0
        elif T[i - 1] < T_off:
            status = 0.0
        u[i] = status
        y = odeint(ruangan, T[i - 1], [ts[i - 1], ts[i]], args=(status,))
        T[i] = y[-1][0]
    energi = np.cumsum(u) * dt / 100 * P_ac          # energi listrik (kWh)
    label = '%d-%d C' % (T_off, T_on)
    plt.subplot(2, 1, 1)
    plt.plot(ts, T, w + '-', label=label)
    plt.subplot(2, 1, 2)
    plt.plot(ts, energi, w + '-', label=label)
    print('Setpoint %s : AC menyala %.2f jam, energi %.2f kWh' %
          (label, u.sum() * dt / 100, energi[-1]))

plt.subplot(2, 1, 1)
plt.plot(ts, T_luar(ts), 'k:', label='suhu luar')
plt.ylabel('Suhu Ruangan $T(t)$, (C)')
plt.legend(frameon=False, ncol=4, loc='lower center', bbox_to_anchor=(0.5, 1.0))
plt.subplot(2, 1, 2)
plt.ylabel('Energi Listrik, (kWh)')
plt.xlabel('$t$, (jam)')
plt.legend(frameon=False)
plt.show()


## Program 14 - Contoh Pemodelan: AC jam 2-7 (GEKKO)

In [ ]:
import numpy as np
from gekko import GEKKO
import matplotlib.pyplot as plt

T_out, tau, k = 32.0, 2.0, 0.07

m = GEKKO(remote=False)
m.time = np.linspace(0, 10, 101)          # titik waktu

# AC menyala jam ke-2 s.d. 7 (GEKKO membaca input di akhir interval, digeser 1 langkah)
u_g = np.zeros(101)
u_g[21:71] = 100.0
u = m.Param(value=u_g)

T = m.Var(30.0)                           # suhu awal 30 C
m.Equation(T.dt() == (T_out - T) / tau - k * u)

m.options.IMODE = 4
m.solve(disp=False)

plt.plot(m.time, T, 'b-')
plt.ylabel('Suhu Ruangan $T(t)$, (C)')
plt.xlabel('$t$, (jam)')
plt.show()
print('GEKKO: T(7)=%.3f  T(10)=%.3f' % (T.value[70], T.value[100]))


## Program 15 - Contoh Pemodelan: variasi koefisien AC k (ODEINT)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

# model suhu ruangan dengan AC
def ruangan(T, t, k, u):
    T_out, tau = 32.0, 2.0
    return (T_out - T) / tau - k * u

ts = np.linspace(0, 10, 101)
u = np.zeros(101)
u[20:70] = 100.0               # AC menyala jam ke-2 s.d. ke-7

# fungsi simulasi untuk nilai k tertentu
def simulasi(k, T0=30.0):
    z = np.zeros(101)
    z[0] = T0
    for i in range(100):
        y = odeint(ruangan, T0, [0, 0.1], args=(k, u[i]))
        T0 = y[-1][0]
        z[i + 1] = T0
    return z

# plot hasil
plt.plot(ts, simulasi(0.03), 'r-', label=r'$k=0.03$')
plt.plot(ts, simulasi(0.05), 'b--', label=r'$k=0.05$')
plt.plot(ts, simulasi(0.07), 'g:', label=r'$k=0.07$')
plt.xlabel('$t$, (jam)')
plt.ylabel('Suhu Ruangan $T(t)$, (C)')
plt.legend()
plt.show()


## Perhitungan manual (Euler) vs analitik vs ODEINT

In [ ]:
import numpy as np
from scipy.integrate import odeint

T_out, tau, k = 32.0, 2.0, 0.07
h = 0.5
t = np.arange(0, 3.0 + h, h)
rows = []
T = 30.0
for ti in t:
    u = 100.0 if 2.0 <= ti < 7.0 else 0.0
    f = (T_out - T) / tau - k * u
    rows.append((ti, u, T, f))
    T = T + h * f

# solusi analitik per segmen
def analitik(tt):
    if tt <= 2:
        return T_out + (30.0 - T_out) * np.exp(-tt / tau)
    T2 = T_out + (30.0 - T_out) * np.exp(-2 / tau)
    Tss = T_out - k * 100 * tau
    return Tss + (T2 - Tss) * np.exp(-(tt - 2) / tau)

def f_ode(T, t):
    u = 100.0 if 2.0 <= t <= 7.0 else 0.0
    return (T_out - T) / tau - k * u
od = odeint(f_ode, 30.0, t, tcrit=[2.0])[:, 0]

print('t | u | T_euler | dT/dt | T_analitik | T_odeint | galat')
for (ti, u, Te, f), o in zip(rows, od):
    a = analitik(ti)
    print('%.1f | %3.0f | %.4f | %+.4f | %.4f | %.4f | %.4f' % (ti, u, Te, f, a, o, abs(Te - a)))
